# triangle-barycentric composite — cx24: combine (u>=0) & (v>=0) & (u+v<=1) into the inside-triangle mask

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `triangle-barycentric`, `boolean-mask-combine`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "triangle-barycentric"
DD_ATOM_IDS = ["triangle-barycentric", "boolean-mask-combine"]
DD_SUBTOPICS = ["Geometry: Barycentric coords", "Numpy: Boolean mask combine"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Once you have `(u, v)` from the barycentric solve, the inside-the-triangle test is THREE predicates AND-ed together: `u >= 0`, `v >= 0`, `u + v <= 1`. Each predicate is a `(N,)` boolean tensor; the combined mask is their elementwise AND.

This is the canonical `boolean-mask-combine` pattern: build the predicates separately (so each is named and debuggable), then `&` them into a single mask. NEVER `|` here — every predicate must hold simultaneously. The pattern composes cleanly with `triangle-barycentric` because the three predicates are exactly the geometric definition of 'point lies inside the triangle in barycentric coordinates'.

### Composite Exercise — combine (u>=0) & (v>=0) & (u+v<=1) into the inside-triangle mask

**Atoms exercised together**: `triangle-barycentric`, `boolean-mask-combine`

Implement `cx24_inside_triangle(u, v)` that takes two `(N,)` tensors of barycentric coordinates and returns a `(N,)` boolean mask: `True` where the corresponding `(u, v)` falls inside the triangle.

Inside-the-triangle: ALL THREE of `u >= 0`, `v >= 0`, `u + v <= 1` must hold.

Build the predicates separately (so each is named) and combine with `&` (NOT `|`). Return the combined boolean mask.

The test covers:
- hand-built `(u, v)` pairs with known inside/outside status,
- the corners of the unit triangle (boundary => True for `>=` / `<=`),
- a random batch cross-checked against `(u >= 0) & (v >= 0) & (u + v <= 1)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx24_inside_triangle(u, v):
    raise NotImplementedError

def _test_cx24():
    # Case A: hand-built points.
    u = t.tensor([0.25,  0.0,  1.0, -0.1, 0.6, 0.5])
    v = t.tensor([0.25,  1.0,  0.0,  0.5, 0.6, 0.4])
    # Inside? (0.25, 0.25) yes; (0, 1) yes (corner); (1, 0) yes (corner);
    # (-0.1, 0.5) no (u<0); (0.6, 0.6) no (u+v=1.2>1); (0.5, 0.4) yes.
    expected = t.tensor([True, True, True, False, False, True])
    mask = cx24_inside_triangle(u, v)
    assert mask.dtype == t.bool, f'expected bool, got {mask.dtype}'
    assert tuple(mask.shape) == tuple(u.shape)
    assert t.equal(mask, expected), f'got {mask}, expected {expected}'

    # Case B: random batch — cross-check against the canonical AND.
    t.manual_seed(13)
    u2 = t.randn(1024) * 0.6
    v2 = t.randn(1024) * 0.6
    mask2 = cx24_inside_triangle(u2, v2)
    ref = (u2 >= 0) & (v2 >= 0) & (u2 + v2 <= 1)
    assert t.equal(mask2, ref), 'mask must match (u>=0) & (v>=0) & (u+v<=1)'

    # Case C: catch the OR-instead-of-AND bug.
    # All-True if you used | because each predicate is True for a different subset.
    u3 = t.tensor([-1.0, 0.5,  2.0])
    v3 = t.tensor([ 0.5, -1.0, 2.0])  # none of these are inside the triangle
    mask3 = cx24_inside_triangle(u3, v3)
    assert not mask3.any(), f'all three points are outside; mask: {mask3}'
    _dd_passed.add('cx24')

_test_cx24()

<details><summary>Show solution — cx24</summary>

```python
def cx24_inside_triangle(u, v):
    # Atom A (triangle-barycentric): a point is inside iff u >= 0, v >= 0, u + v <= 1.
    p_u = u >= 0
    p_v = v >= 0
    p_sum = (u + v) <= 1
    # Atom B (boolean-mask-combine): elementwise AND across the three predicates.
    return p_u & p_v & p_sum
```

Building the three predicates as named tensors first (rather than one big chained expression) makes each easy to inspect at debug time — you can `mask.any()` / `mask.sum()` per predicate to see which constraint is firing. The combining operator is `&` (elementwise AND): `|` would give the OR of the three regions, which is the WHOLE plane minus the third-quadrant + the u+v>1 wedge — emphatically not the triangle interior.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx24'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx24',
        'subtopics': ["Geometry: Barycentric coords", "Numpy: Boolean mask combine"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()